#Data Quality Analysis
**Missing Values**: Significant gaps were found in mobility metrics, specifically in the retail_recreation, workplaces, and residential columns. These missing entries could distort our correlation analysis if not handled

**Presence of Outliers**: The daily_cases column shows an extreme range—varying from 0 to over 11,000 cases. With a mean of only 71, it's clear that a few extreme spikes (outliers) are present, likely from high-population centers or reporting anomalies

**Format Inconsistency**: The date column was imported as a string object, which prevents time-series plotting and grouping until it is converted to a proper datetime format

#Missing Value Strategy
**Strategy**: Listwise Deletion (Dropping rows with null values).
With over 20,000 records, removing rows with missing mobility data preserves the integrity of real-world observations

In [ ]:
# Cleaning missing values
df_clean = df.dropna(subset=['retail_recreation', 'workplaces', 'residential'])
print(f"Cleaned dataset shape: {df_clean.shape}")

#Outlier Handling (IQR Method)
To prevent extreme case spikes from skewing the results, we filter the daily_cases column using the Interquartile Range

In [ ]:
# Detect and handle outliers
Q1 = df_clean['daily_cases'].quantile(0.25)
Q3 = df_clean['daily_cases'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df_filtered = df_clean[(df_clean['daily_cases'] >= lower_bound) &
                        (df_clean['daily_cases'] <= upper_bound)]

print(f"Records after outlier removal: {df_filtered.shape[0]}")

#Numerical Feature Normalization
Normalization is required to bring features like **cases** and **mobility percentages** onto the same scale for PCA

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

cols = ['daily_cases', 'retail_recreation', 'workplaces', 'residential']
data = df_filtered[cols]

# Min-Max Normalization (0 to 1 range)
scaler_mm = MinMaxScaler()
df_minmax = pd.DataFrame(scaler_mm.fit_transform(data), columns=cols)

# Z-score Normalization (Mean 0, Std Dev 1)
scaler_std = StandardScaler()
df_zscore = pd.DataFrame(scaler_std.fit_transform(data), columns=cols)

df_zscore.head()

#PCA and Explained Variance
Applying Principal Component Analysis to reduce dimensionality and identify the primary drivers of data variance.

In [ ]:
from sklearn.decomposition import PCA

# Apply PCA
pca = PCA()
pca_results = pca.fit_transform(df_zscore)

# Interpretation
var_ratio = pca.explained_variance_ratio_
for i, v in enumerate(var_ratio):
    print(f"PC{i+1} Explained Variance: {v:.2%}")

# Plotting
plt.bar(range(1, 5), var_ratio, color='steelblue')
plt.title('Explained Variance by Component')
plt.xlabel('Principal Component')
plt.ylabel('Variance Ratio')
plt.show()